# Vectron AI - One-Click Colab Training

End-to-end pipeline:

1. Install dependencies
2. Clone the Vectron-AI repository
3. Download a small Iconify subset
4. Clean + caption + build dataset
5. QLoRA fine-tune `Qwen2.5-Coder-0.5B-Instruct`
6. Generate a sample SVG
7. (Optional) Save the LoRA adapter to Google Drive

**Recommended runtime:** `Runtime -> Change runtime type -> T4 GPU` (free).

On a free T4, training on ~5000 samples for 1 epoch takes ~30 minutes.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo and install dependencies

In [ ]:
import os
if not os.path.exists('Vectron-AI'):
    !git clone https://github.com/hindiakshar0702-star/Vectron-AI.git
%cd Vectron-AI

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python scripts/check_environment.py

## 3. (Optional) Sign in to Hugging Face and W&B

Skip these cells if you don't want experiment tracking.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# import wandb
# wandb.login()

## 4. Download a small Iconify subset

We start with 5000 icons from Lucide + Heroicons + Tabler. This is plenty for a first PoC and finishes in ~2 minutes.

In [ ]:
!python data/download_iconify.py \
    --collections lucide,heroicons,tabler \
    --output data/raw \
    --max-icons 5000

## 5. Clean SVGs

In [ ]:
!python data/svg_cleaner.py --input data/raw --output data/clean --target-size 24

## 6. Generate captions (heuristic mode - no GPU needed)

In [ ]:
!python data/caption_generator.py \
    --input data/clean \
    --output data/captions.jsonl \
    --mode heuristic

## 7. Build the training JSONL

In [ ]:
!python data/build_dataset.py \
    --svg-dir data/clean \
    --captions data/captions.jsonl \
    --output data/train.jsonl \
    --eval-output data/eval.jsonl \
    --eval-fraction 0.02

In [ ]:
!head -n 1 data/train.jsonl | python -m json.tool | head -n 40

## 8. Train with the `colab_t4` preset

This uses `Qwen2.5-Coder-0.5B-Instruct` with QLoRA so it fits in T4's 16 GB VRAM.

In [ ]:
!python -m training.train_lora \
    --config training/config.yaml \
    --preset colab_t4

## 9. Generate a test SVG

In [ ]:
from inference.generate import GenerationConfig, VectronGenerator
from inference.postprocess import postprocess
from inference.render import svg_to_pil

cfg = GenerationConfig(
    base_model='Qwen/Qwen2.5-Coder-0.5B-Instruct',
    adapter='checkpoints/vectron-v0.1/final',
    num_variants=4,
    temperature=0.8,
)
gen = VectronGenerator(cfg)
raw = gen.generate('minimal blue wallet icon, fintech, flat')
for i, r in enumerate(raw):
    out = postprocess(r)
    print(f'variant {i}: valid={out.valid} issues={out.issues}')
    if out.valid:
        display(svg_to_pil(out.svg, width=128, height=128))

## 10. (Optional) Save the LoRA adapter to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r checkpoints/vectron-v0.1 /content/drive/MyDrive/vectron/

## Next steps

- Scale up to 50K+ icons (`--collections mdi,ph,carbon,fluent,solar`).
- Switch preset to `colab_a100` or `runpod_a100` for the 1.5B / 7B models.
- Caption with BLIP-2 for richer prompts: `python data/caption_generator.py --mode blip2`.
- Push the adapter to Hugging Face: `huggingface-cli upload <user>/vectron-v0.1 checkpoints/vectron-v0.1/final`.
- Serve with FastAPI: `uvicorn api.server:app --port 8000`.